# Observation methods

monosemantic 
multisemantic

### Probing (2019)
[[paper]](https://aclanthology.org/N19-1419.pdf)
<br>__Idea:__ train a lightweight classifier to test whether a property (POS tags, syntax depth, etc.) is linearly readable from hidden states 

### Logit Lens (2020)
[[paper]](https://arxiv.org/pdf/2303.08112)
<br> __Idea:__ At each layer project intermediate outputs into vocab space using fixed final layer projection
<br>Motivation = to see “the model’s current guess” at each layer

### Tuned Lens (2023)
[[paper]](https://arxiv.org/pdf/2303.08112)
<br>__Idea:__ The same as LogitLens but they add additional trainable layer before each projection. It is trained as adapter on a frozen model
<br>Motivation = to see “what model could potentially predict at each layer”

### Direct Logit Attribution
[[paper]](https://arxiv.org/pdf/2303.08112)
<br>__Idea:__ Linearly attribute final logits to contributions from specific components (heads/MLPs/positions); useful “who pushed the logit?” readout, but be cautious about LayerNorm/linearization assumptions and adversarial edge cases

### Attention Head Analysis (2022)
[[paper]](https://arxiv.org/pdf/2211.00593)
<br>Inspect attention maps & head roles (e.g., induction heads, name-mover heads) and how they compose into circuits on concrete tasks like IOI

### Representation Similarity (1997)
[[paper]](https://proceedings.mlr.press/v97/kornblith19a/kornblith19a.pdf)
<br>A family of methods to compare layer outputs (i.e. SVCCA/CKA/RSA)

Comparison can be done:
- between different layers (across model depth)
- between corresponding layers of two models
- between model checkpoints (across training completeness)

It allows to see the signal dynamics / possible layer redundancy / possible phase shift in training process etc

### Patchscopes (2024)
[[paper]](https://arxiv.org/abs/2401.06102)
<br>Idea: copy-paste weight to another model and see how it affects the answer

Algorithm:
- do "source" run
- prepare target run: construct a prompt with a placeholder
- copy-paste inspected weight to the desired position (layer, placeholder)
- do target run
- analyze what was generated

### Sparse Autoencoders (2024)
[[paper]](https://arxiv.org/abs/2406.04093)
<br>__Idea:__ try to build a sparse representation of intermediate signal - patterns

Algorithm:
- choose the layer to inspect
- freeze the model and train SAE
- intermediate state of SAE represents sparse “features” that fire on monosemantic concepts

Top-K<br>
Dictionary Learning



Some patterns in Attention Heads

| Pattern Name                   | What It Looks Like                                                                         | Likely Function                                                                                         |
| ------------------------------ | ------------------------------------------------------------------------------------------ | ------------------------------------------------------------------------------------------------------- |
| **Induction Heads**            | Strong diagonal attention from a token to its *previous occurrence* earlier in the context | Enables “copy after repetition” — a mechanism for few-shot learning and in-context pattern continuation |
| **Duplicate Token Heads**      | Attend to the *nearest identical token* in the past                                        | Support repetition and exact match completion                                                           |
| **Name Mover Heads**           | In question-answer tasks, jump from the question name to the answer name position          | Important in IOI (Indirect Object Identification) circuits                                              |
| **Next Token Heads**           | Attend to the *immediately previous token*                                                 | Acts like a basic Markov chain, helping next-token prediction in local contexts                         |
| **Syntactic Heads**            | Focus on fixed dependency relations (e.g., subject → verb, noun → determiner)              | Encodes grammatical structure                                                                           |
| **Bridging Heads**             | Attend from a pronoun or anaphor to its referent                                           | Supports coreference resolution                                                                         |
| **Positional Heads**           | Almost entirely determined by relative positions, regardless of token identity             | Often act as position encoders or help combine local context windows                                    |
| **Stopword Suppression Heads** | Attend away from common stopwords                                                          | Help downstream components ignore low-information tokens                                                |
| **Translation Heads**          | In multilingual models, attend to semantically equivalent words in different languages     | Assist in cross-lingual alignment                                                                       |
| **Copy Heads**                 | Nearly uniform attention to a specific previous position                                   | Support literal copying of subsequences                                                                 |


# Intervention

### Activation Patching (a.k.a. causal tracing / interchange intervention). 

Swap activations between clean and corrupted runs at chosen nodes (layers/heads/positions) and see if behavior recovers—localizes which parts are sufficient/necessary for a behavior. Details (metric, corruption choice, LayerNorm handling) really matter

### Attribution Patching (approximate, gradient-based). 

A fast proxy for exhaustive patching: estimate all potential patches with two forward passes + one backward pass, then verify promising nodes with real patching (e.g., AtP*). Great for pre-filtering large graphs

### Path Patching (edge-level). 
Patch only along hypothesized edges (source→target) to distinguish direct effects from mediation and map minimal circuits. 

### Causal Scrubbing (you probably meant this; not “casual scrabbing”). 

Formal hypothesis testing via behavior-preserving resampling ablations—verify whether a proposed circuit/mechanism actually explains the behavior

### Weight-Level Knowledge Editing (ROME/MEMIT). 

Perform rank-one (ROME) or mass (MEMIT) edits to MLP weights to change specific factual associations; useful to test where knowledge lives and how edits ripple

### Component Knockouts/Ablations. 

Zero or remove specific heads/MLPs/positions to see causal impact; often used inside circuit case studies like IOI. 
OpenReview

### Causal Mediation Analysis. 

Estimate indirect effects via mediators (heads/neurons) on outputs; applied to, e.g., gender-bias flow and arithmetic behaviors. 
NeurIPS Proceedings

### Activation/Feature Steering (Activation Addition / CAA, SAE-guided). 

Add learned or contrastive “steering vectors” (optionally in SAE-feature space) to hidden states at inference to causally shift behavior—useful both for control and validating learned features